In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1: Install dependencies (RUN FIRST)
# ═══════════════════════════════════════════════════════════
%pip -q install torch torchvision torchaudio
%pip -q install transformers timm huggingface_hub safetensors
%pip -q install scikit-learn scipy statsmodels pandas numpy matplotlib seaborn tqdm tabulate

# 🔬 DR Frozen-Encoder Benchmark
## MedSigLIP vs RETFound vs EfficientNet-B0
**Calibration-focused, contamination-aware evaluation on APTOS (dev) + MESSIDOR-2 (external)**

PRD Version 1.0 — Companion notebook for manuscript submission

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2: Runtime setup, Google Drive mount, directory structure
# ═══════════════════════════════════════════════════════════
import os
from pathlib import Path

import torch

IS_COLAB = "google.colab" in str(type(get_ipython()))  # noqa: F821
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise EnvironmentError(
        "GPU is required. Go to Runtime > Change runtime type > T4 GPU."
    )

# ── Google Drive mount ──
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/DR_Benchmark")
else:
    DRIVE_ROOT = Path("./DR_Benchmark")  # local fallback

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        f"Drive root not found: {DRIVE_ROOT}\n"
        "Create 'DR_Benchmark' folder in Google Drive with:\n"
        "  train_aptos.csv\n"
        "  train_images_aptos/  (3,662 PNG)\n"
        "  messidor_data.csv\n"
        "  IMAGES_messidor/  (1,748 PNG)"
    )

# ── HuggingFace token (needed for gated models) ──
from google.colab import userdata

try:
    # Colab Secrets (Sol taraftaki anahtar simgesi) üzerinden okur
    HF_TOKEN = userdata.get('HF_TOKEN').strip()
except Exception:
    # Eğer Secrets'ta yoksa os.environ'a bakar, o da yoksa boş string döner
    HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    print("[WARN] HF_TOKEN not set. Run: import os; os.environ['HF_TOKEN']='hf_...'")
    print("       Both MedSigLIP and RETFound are GATED models.")
    print("       1. Create HF token at https://huggingface.co/settings/tokens")
    print("       2. Accept license at https://huggingface.co/google/medsiglip-448")
    print("       3. Accept license at https://huggingface.co/YukunZhou/RETFound_mae_natureCFP")

# ── Working directories ──
ROOT = Path("/content")
ARTIFACT_ROOT = ROOT / "artifacts"

for p in [
    ARTIFACT_ROOT / "metadata",
    ARTIFACT_ROOT / "embeddings",
    ARTIFACT_ROOT / "splits",
    ARTIFACT_ROOT / "checkpoints",
    ARTIFACT_ROOT / "predictions",
    ARTIFACT_ROOT / "metrics",
    ARTIFACT_ROOT / "tables",
    ARTIFACT_ROOT / "figures",
    ARTIFACT_ROOT / "supplementary",
    ARTIFACT_ROOT / "export",
]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Device       : {DEVICE}")
print(f"Drive root   : {DRIVE_ROOT}")
print(f"Artifacts    : {ARTIFACT_ROOT}")
print(f"HF_TOKEN set : {bool(HF_TOKEN)}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3: Imports + experiment configuration
# ═══════════════════════════════════════════════════════════
import gc
import json
import time
import random
import zipfile
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from tqdm.auto import tqdm

from scipy.special import softmax
from scipy.optimize import minimize_scalar
from scipy import stats

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    brier_score_loss,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    cohen_kappa_score,
    roc_curve,
    confusion_matrix,
)

from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

from transformers import AutoModel, AutoProcessor
from huggingface_hub import hf_hub_download, list_repo_files

import timm
import torchvision
from torchvision import transforms

warnings.filterwarnings("ignore", category=UserWarning)

# ── Publication-quality plot defaults ──
matplotlib.rcParams.update({
    "font.family": "Arial",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})
sns.set_theme(style="whitegrid", font="Arial")


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


@dataclass
class ExperimentConfig:
    # ── Profile ──
    profile: str = "paper"
    seeds: Tuple[int, ...] = (13, 17, 23, 29, 31)
    split_seed: int = 42
    # ── Data schema ──
    aptos_required_cols: Tuple[str, ...] = ("id_code", "diagnosis")
    # ── Training ──
    batch_size: int = 64
    num_workers: int = 2
    max_epochs: int = 30
    patience: int = 5
    lr: float = 1e-3
    weight_decay: float = 1e-2
    dropout: float = 0.1
    hidden_dim: int = 512
    n_splits: int = 5
    val_ratio: float = 0.15
    # ── Evaluation ──
    ece_bins: int = 10
    bootstrap_iters: int = 1000
    # ── Model IDs ──
    medsiglip_hf_id: str = "google/medsiglip-448"
    retfound_hf_id: str = "YukunZhou/RETFound_mae_natureCFP"
    retfound_arch: str = "vit_large_patch16_224"
    efficientnet_name: str = "efficientnet_b0"
    # ── Optional analyses ──
    run_ablations: bool = True
    run_robustness: bool = False


cfg = ExperimentConfig()
set_global_seed(cfg.split_seed)

MODEL_REGISTRY: Dict[str, Dict[str, Any]] = {
    "medsiglip":      {"embed_dim": 1152, "input_size": 448, "kind": "hf_siglip"},
    "retfound":       {"embed_dim": 1024, "input_size": 224, "kind": "retfound_mae"},
    "efficientnet_b0": {"embed_dim": 1280, "input_size": 224, "kind": "efficientnet"},
}

print("Config loaded:")
print(json.dumps(asdict(cfg), indent=2))

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 4: Load data from Google Drive (no upload, no extraction)
# ═══════════════════════════════════════════════════════════

# ── Path definitions ──
APTOS_IMG_ROOT = DRIVE_ROOT / "train_images_aptos"
APTOS_CSV_PATH = DRIVE_ROOT / "train_aptos.csv"
MESSIDOR_IMG_ROOT = DRIVE_ROOT / "IMAGES_messidor"
MESSIDOR_CSV_PATH = DRIVE_ROOT / "messidor_data.csv"

# ── Verify all paths exist ──
for label, path in [
    ("APTOS images", APTOS_IMG_ROOT),
    ("APTOS CSV", APTOS_CSV_PATH),
    ("MESSIDOR images", MESSIDOR_IMG_ROOT),
    ("MESSIDOR CSV", MESSIDOR_CSV_PATH),
]:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found at: {path}")
    print(f"[OK] {label}: {path}")

# ── Count images ──
aptos_images = list(APTOS_IMG_ROOT.glob("*.png"))
messidor_images = list(MESSIDOR_IMG_ROOT.glob("*.png")) + list(MESSIDOR_IMG_ROOT.glob("*.jpg")) + list(MESSIDOR_IMG_ROOT.glob("*.JPG"))
print(f"\nAPTOS images found  : {len(aptos_images)}")
print(f"MESSIDOR images found: {len(messidor_images)}")

if len(aptos_images) < 3600:
    raise ValueError(f"Expected ~3,662 APTOS images, found {len(aptos_images)}")
if len(messidor_images) < 1700:
    raise ValueError(f"Expected ~1,748 MESSIDOR images, found {len(messidor_images)}")

print("\nAll data paths verified.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 5: Schema validation, column mapping, class distributions
# ═══════════════════════════════════════════════════════════

# ── APTOS ──
aptos_df = pd.read_csv(APTOS_CSV_PATH)
missing = [c for c in cfg.aptos_required_cols if c not in aptos_df.columns]
if missing:
    raise ValueError(f"APTOS CSV missing columns: {missing}. Found: {list(aptos_df.columns)}")

aptos_df = aptos_df.copy()
aptos_df["grade"] = aptos_df["diagnosis"].astype(int)
aptos_df["binary_label"] = (aptos_df["grade"] >= 2).astype(int)

# Build image index (stem → path)
aptos_img_index = {p.stem: p for p in APTOS_IMG_ROOT.glob("*.png")}
aptos_df["image_path"] = aptos_df["id_code"].apply(
    lambda x: str(aptos_img_index.get(str(x), "NOT_FOUND"))
)
n_missing = (aptos_df["image_path"] == "NOT_FOUND").sum()
if n_missing > 0:
    print(f"[WARN] {n_missing} APTOS images not found — dropping these rows.")
    aptos_df = aptos_df[aptos_df["image_path"] != "NOT_FOUND"].reset_index(drop=True)
aptos_df["dataset"] = "aptos"

# ── MESSIDOR-2 (Google-3 adjudicated grades) ──
messidor_df = pd.read_csv(MESSIDOR_CSV_PATH)
print(f"MESSIDOR CSV columns: {list(messidor_df.columns)}")

# Column mapping: Kaggle Google-3 CSV uses 'adjudicated_dr_grade'
MESSIDOR_COL_MAP = {
    "adjudicated_dr_grade": "grade",
    "adjudicated_dme": "dme",
    "adjudicated_gradable": "gradable",
}
messidor_df = messidor_df.rename(columns=MESSIDOR_COL_MAP)

# Validate required columns
for col in ["image_id", "grade"]:
    if col not in messidor_df.columns:
        raise ValueError(
            f"MESSIDOR CSV missing '{col}' after mapping. "
            f"Available: {list(messidor_df.columns)}. "
            f"Adjust MESSIDOR_COL_MAP if your CSV has different column names."
        )

# Filter: keep only gradable images
if "gradable" in messidor_df.columns:
    n_before = len(messidor_df)
    messidor_df = messidor_df[messidor_df["gradable"] == True].reset_index(drop=True)
    n_dropped = n_before - len(messidor_df)
    if n_dropped > 0:
        print(f"[INFO] Dropped {n_dropped} ungradable MESSIDOR images ({n_before} → {len(messidor_df)})")

messidor_df = messidor_df.copy()
messidor_df["grade"] = messidor_df["grade"].astype(int)
if not messidor_df["grade"].between(0, 4).all():
    raise ValueError("MESSIDOR grade values must be in [0, 4].")
messidor_df["binary_label"] = (messidor_df["grade"] >= 2).astype(int)

# Build image index
messidor_img_index = {p.stem: p for p in list(MESSIDOR_IMG_ROOT.glob("*.png")) + list(MESSIDOR_IMG_ROOT.glob("*.jpg")) + list(MESSIDOR_IMG_ROOT.glob("*.JPG"))}

def resolve_messidor_id(image_id: str) -> str:
    """Match image_id from CSV to actual file on disk."""
    key = str(image_id).strip()
    # Try exact stem match
    if key in messidor_img_index:
        return str(messidor_img_index[key])
    # Try without extension if CSV has extension
    stem = Path(key).stem
    if stem in messidor_img_index:
        return str(messidor_img_index[stem])
    return "NOT_FOUND"

messidor_df["image_path"] = messidor_df["image_id"].apply(resolve_messidor_id)
n_missing = (messidor_df["image_path"] == "NOT_FOUND").sum()
if n_missing > 0:
    print(f"[WARN] {n_missing}/{len(messidor_df)} MESSIDOR images not matched.")
    print("  Sample unmatched IDs:", messidor_df[messidor_df["image_path"] == "NOT_FOUND"]["image_id"].head(5).tolist())
    print("  Sample filenames on disk:", list(messidor_img_index.keys())[:5])
    messidor_df = messidor_df[messidor_df["image_path"] != "NOT_FOUND"].reset_index(drop=True)
    print(f"  Remaining after drop: {len(messidor_df)}")
messidor_df["dataset"] = "messidor"

# ── Save clean CSVs ──
aptos_df.to_csv(ARTIFACT_ROOT / "metadata" / "aptos_clean.csv", index=False)
messidor_df.to_csv(ARTIFACT_ROOT / "metadata" / "messidor_clean.csv", index=False)

# ── MESSIDOR-2 patient-level mapping (from official left/right eye pairing) ──
PATIENT_MAP_PATH = DRIVE_ROOT / "messidor_patient_mapping_official.csv"
if PATIENT_MAP_PATH.exists():
    patient_map = pd.read_csv(PATIENT_MAP_PATH)
    messidor_df = messidor_df.merge(
        patient_map[["image_id", "patient_id", "laterality"]],
        on="image_id", how="left"
    )
    n_mapped = messidor_df["patient_id"].notna().sum()
    print(f"[INFO] Patient mapping: {n_mapped}/{len(messidor_df)} images matched to {messidor_df['patient_id'].nunique()} patients")
else:
    # Fallback: no patient mapping available
    messidor_df["patient_id"] = messidor_df["image_id"]
    messidor_df["laterality"] = "unknown"
    print("[WARN] Patient mapping CSV not found — treating each image as independent")

# ── Alternative binary label for sensitivity analysis (M5: grade >= 1) ──
aptos_df["binary_label_alt"] = (aptos_df["grade"] >= 1).astype(int)
messidor_df["binary_label_alt"] = (messidor_df["grade"] >= 1).astype(int)

# ── Print summaries ──
print(f"\n{'='*50}")
print(f"APTOS: {len(aptos_df)} images")
print("  5-class:", aptos_df["grade"].value_counts().sort_index().to_dict())
print("  Binary :", aptos_df["binary_label"].value_counts().sort_index().to_dict())
print(f"\nMESSIDOR-2: {len(messidor_df)} images")
print("  5-class:", messidor_df["grade"].value_counts().sort_index().to_dict())
print("  Binary :", messidor_df["binary_label"].value_counts().sort_index().to_dict())
print(f"{'='*50}")
print("  Binary (grade>=1):", aptos_df["binary_label_alt"].value_counts().sort_index().to_dict())
# ... ve MESSIDOR için aynısı
print("  Binary (grade>=1):", messidor_df["binary_label_alt"].value_counts().sort_index().to_dict())

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6: Model loaders + embedding extraction API
# ═══════════════════════════════════════════════════════════

def get_medsiglip_bundle() -> Dict[str, Any]:
    processor = AutoProcessor.from_pretrained(cfg.medsiglip_hf_id, token=HF_TOKEN or None)
    model = AutoModel.from_pretrained(cfg.medsiglip_hf_id, token=HF_TOKEN or None)
    model.eval().to(DEVICE)
    # Verify embedding dim
    dummy = processor(images=[Image.new("RGB", (448, 448))], return_tensors="pt")
    dummy = {k: v.to(DEVICE) for k, v in dummy.items()}
    with torch.inference_mode():
        vision_out = model.vision_model(pixel_values=dummy["pixel_values"])
        test_emb = vision_out.pooler_output
    actual_dim = test_emb.shape[1]
    expected_dim = MODEL_REGISTRY["medsiglip"]["embed_dim"]
    if actual_dim != expected_dim:
        print(f"[AUTO-FIX] MedSigLIP embed_dim: expected {expected_dim}, got {actual_dim}. Updating registry.")
        MODEL_REGISTRY["medsiglip"]["embed_dim"] = actual_dim
    print(f"MedSigLIP loaded. Embed dim: {actual_dim}")
    del dummy, test_emb
    torch.cuda.empty_cache()
    return {"processor": processor, "model": model}


def _extract_state_dict(checkpoint_obj: Any) -> Dict[str, torch.Tensor]:
    if isinstance(checkpoint_obj, dict):
        for key in ["state_dict", "model", "module", "teacher", "student"]:
            if key in checkpoint_obj and isinstance(checkpoint_obj[key], dict):
                return checkpoint_obj[key]
        if all(isinstance(v, torch.Tensor) for v in checkpoint_obj.values()):
            return checkpoint_obj
    raise ValueError("Could not extract state_dict from RETFound checkpoint.")


def _clean_vit_state_dict(sd: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    cleaned = {}
    for k, v in sd.items():
        nk = k
        for prefix in ["module.", "backbone.", "encoder."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v
    return cleaned


def get_retfound_bundle() -> Dict[str, Any]:
    files = list_repo_files(cfg.retfound_hf_id, token=HF_TOKEN or None)
    print(f"RETFound repo files: {[f for f in files if f.endswith(('.pth', '.pt', '.bin', '.safetensors'))]}")
    preferred = ["RETFound_cfp_weights.pth", "pytorch_model.bin", "model.safetensors", "checkpoint.pth"]
    candidate = None
    for f in preferred:
        if f in files:
            candidate = f
            break
    if candidate is None:
        pth_candidates = [f for f in files if f.endswith((".pth", ".pt", ".bin", ".safetensors"))]
        if not pth_candidates:
            raise FileNotFoundError(f"No checkpoint file found in {cfg.retfound_hf_id}. Files: {files}")
        candidate = pth_candidates[0]
    print(f"Downloading RETFound checkpoint: {candidate}")

    local_ckpt = hf_hub_download(cfg.retfound_hf_id, filename=candidate, token=HF_TOKEN or None)
    model = timm.create_model(cfg.retfound_arch, pretrained=False, num_classes=0, global_pool="avg")
    if candidate.endswith(".safetensors"):
        from safetensors.torch import load_file
        sd = load_file(local_ckpt)
    else:
        ckpt = torch.load(local_ckpt, map_location="cpu", weights_only=False)
        sd = _extract_state_dict(ckpt)

    sd = _clean_vit_state_dict(sd)
    # Filter out head/decoder weights
    model_keys = set(model.state_dict().keys())
    sd_filtered = {k: v for k, v in sd.items() if k in model_keys}
    msg = model.load_state_dict(sd_filtered, strict=False)
    print(f"RETFound load: missing={len(msg.missing_keys)}, unexpected={len(msg.unexpected_keys)}")
    if msg.missing_keys:
        print(f"  Missing (first 5): {msg.missing_keys[:5]}")

    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    model.eval().to(DEVICE)

    # Verify embedding dim
    with torch.inference_mode():
        test_emb = model(torch.randn(1, 3, 224, 224).to(DEVICE))
    actual_dim = test_emb.shape[1]
    expected_dim = MODEL_REGISTRY["retfound"]["embed_dim"]
    if actual_dim != expected_dim:
        print(f"[AUTO-FIX] RETFound embed_dim: expected {expected_dim}, got {actual_dim}. Updating registry.")
        MODEL_REGISTRY["retfound"]["embed_dim"] = actual_dim
    print(f"RETFound loaded. Embed dim: {actual_dim}")
    del test_emb
    torch.cuda.empty_cache()
    return {"preprocess": preprocess, "model": model}


def get_efficientnet_bundle() -> Dict[str, Any]:
    weights = torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1
    model = torchvision.models.efficientnet_b0(weights=weights)
    model.classifier = nn.Identity()
    preprocess = weights.transforms()
    model.eval().to(DEVICE)
    # Verify
    with torch.inference_mode():
        test_emb = model(torch.randn(1, 3, 224, 224).to(DEVICE))
    actual_dim = test_emb.shape[1]
    expected_dim = MODEL_REGISTRY["efficientnet_b0"]["embed_dim"]
    if actual_dim != expected_dim:
        print(f"[AUTO-FIX] EfficientNet embed_dim: expected {expected_dim}, got {actual_dim}. Updating registry.")
        MODEL_REGISTRY["efficientnet_b0"]["embed_dim"] = actual_dim
    print(f"EfficientNet-B0 loaded. Embed dim: {actual_dim}")
    del test_emb
    torch.cuda.empty_cache()
    return {"preprocess": preprocess, "model": model}


LOADER_REGISTRY = {
    "medsiglip": get_medsiglip_bundle,
    "retfound": get_retfound_bundle,
    "efficientnet_b0": get_efficientnet_bundle,
}


@torch.inference_mode()
def _forward_embeddings(model_key: str, bundle: Dict[str, Any], pil_images: List[Image.Image]) -> np.ndarray:
    if model_key == "medsiglip":
        processor = bundle["processor"]
        model = bundle["model"]
        inputs = processor(images=pil_images, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.cuda.amp.autocast(enabled=True):
            vision_out = model.vision_model(pixel_values=inputs["pixel_values"])
            emb = vision_out.pooler_output
        return emb.detach().float().cpu().numpy()

    preprocess = bundle["preprocess"]
    model = bundle["model"]
    x = torch.stack([preprocess(img) for img in pil_images]).to(DEVICE)
    with torch.cuda.amp.autocast(enabled=True):
        out = model(x)
    if out.ndim > 2:
        out = out.mean(dim=(-2, -1))
    return out.detach().float().cpu().numpy()


def extract_embeddings(model_key: str, dataset_df: pd.DataFrame, out_prefix: Path, id_col: str, label_col: str) -> Dict[str, Path]:
    out_prefix.parent.mkdir(parents=True, exist_ok=True)
    emb_path = Path(str(out_prefix) + "_embeddings.npy")
    ids_path = Path(str(out_prefix) + "_ids.npy")
    labels_path = Path(str(out_prefix) + "_labels.npy")
    if emb_path.exists() and ids_path.exists() and labels_path.exists():
        emb = np.load(emb_path, mmap_mode="r")
        expected_dim = MODEL_REGISTRY[model_key]["embed_dim"]
        if emb.shape == (len(dataset_df), expected_dim):
            print(f"[CACHE] {model_key} -> {emb_path} ({emb.shape})")
            return {"embeddings": emb_path, "ids": ids_path, "labels": labels_path}
        else:
            print(f"[CACHE INVALID] Shape mismatch: {emb.shape} vs expected ({len(dataset_df)}, {expected_dim}). Re-extracting.")

    bundle = LOADER_REGISTRY[model_key]()
    batch_size = cfg.batch_size
    n = len(dataset_df)
    all_emb, all_ids, all_labels = [], [], []

    for start in tqdm(range(0, n, batch_size), desc=f"Extract {model_key}"):
        stop = min(start + batch_size, n)
        batch = dataset_df.iloc[start:stop]
        pil_images = [Image.open(p).convert("RGB") for p in batch["image_path"].tolist()]
        emb = _forward_embeddings(model_key, bundle, pil_images)
        all_emb.append(emb.astype(np.float32))
        all_ids.extend(batch[id_col].astype(str).tolist())
        all_labels.extend(batch[label_col].astype(int).tolist())

    emb_arr = np.concatenate(all_emb, axis=0)
    expected_dim = MODEL_REGISTRY[model_key]["embed_dim"]
    if emb_arr.shape != (n, expected_dim):
        raise ValueError(f"Shape mismatch for {model_key}: got {emb_arr.shape}, expected ({n}, {expected_dim})")

    np.save(emb_path, emb_arr)
    np.save(ids_path, np.array(all_ids, dtype=object))
    np.save(labels_path, np.array(all_labels, dtype=np.int64))

    del bundle
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Saved: {model_key} embeddings {emb_arr.shape}")
    return {"embeddings": emb_path, "ids": ids_path, "labels": labels_path}


print("Model loaders and extraction API ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7: Extract MedSigLIP embeddings (APTOS + MESSIDOR)
# ═══════════════════════════════════════════════════════════
EMBED_INDEX: Dict[str, Dict[str, Dict[str, Path]]] = {}
model_key = "medsiglip"
EMBED_INDEX[model_key] = {}
EMBED_INDEX[model_key]["aptos"] = extract_embeddings(
    model_key=model_key, dataset_df=aptos_df,
    out_prefix=ARTIFACT_ROOT / "embeddings" / model_key / "aptos",
    id_col="id_code", label_col="grade",
)
EMBED_INDEX[model_key]["messidor"] = extract_embeddings(
    model_key=model_key, dataset_df=messidor_df,
    out_prefix=ARTIFACT_ROOT / "embeddings" / model_key / "messidor",
    id_col="image_id", label_col="grade",
)
print(f"MedSigLIP done: {EMBED_INDEX[model_key]}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 8: Extract RETFound embeddings (APTOS + MESSIDOR)
# ═══════════════════════════════════════════════════════════
if not HF_TOKEN:
    print("[WARN] HF_TOKEN not set. If RETFound is gated, this will fail.")
model_key = "retfound"
EMBED_INDEX[model_key] = {}
EMBED_INDEX[model_key]["aptos"] = extract_embeddings(
    model_key=model_key, dataset_df=aptos_df,
    out_prefix=ARTIFACT_ROOT / "embeddings" / model_key / "aptos",
    id_col="id_code", label_col="grade",
)
EMBED_INDEX[model_key]["messidor"] = extract_embeddings(
    model_key=model_key, dataset_df=messidor_df,
    out_prefix=ARTIFACT_ROOT / "embeddings" / model_key / "messidor",
    id_col="image_id", label_col="grade",
)
print(f"RETFound done: {EMBED_INDEX[model_key]}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 9: Extract EfficientNet-B0 embeddings (APTOS + MESSIDOR)
# ═══════════════════════════════════════════════════════════
model_key = "efficientnet_b0"
EMBED_INDEX[model_key] = {}
EMBED_INDEX[model_key]["aptos"] = extract_embeddings(
    model_key=model_key, dataset_df=aptos_df,
    out_prefix=ARTIFACT_ROOT / "embeddings" / model_key / "aptos",
    id_col="id_code", label_col="grade",
)
EMBED_INDEX[model_key]["messidor"] = extract_embeddings(
    model_key=model_key, dataset_df=messidor_df,
    out_prefix=ARTIFACT_ROOT / "embeddings" / model_key / "messidor",
    id_col="image_id", label_col="grade",
)
print(f"EfficientNet-B0 done: {EMBED_INDEX[model_key]}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 10: Build stratified 5-fold CV splits with fold-internal validation
# ═══════════════════════════════════════════════════════════
def build_splits(df, task, n_splits=5, val_ratio=0.15):
    if task == "binary":
        y = df["binary_label"].values
    elif task == "binary_alt":
        y = df["binary_label_alt"].values
    else:
        y = df["grade"].values
    # ... gerisi aynı
    indices = np.arange(len(df))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=cfg.split_seed)
    folds = []
    for fold_id, (train_idx, test_idx) in enumerate(skf.split(indices, y)):
        tr_idx, val_idx = train_test_split(
            train_idx, test_size=val_ratio, stratify=y[train_idx],
            random_state=cfg.split_seed + fold_id,
        )
        folds.append({"fold": fold_id, "train_idx": tr_idx.tolist(), "val_idx": val_idx.tolist(), "test_idx": test_idx.tolist()})

    # Sanity check: test folds must be a perfect partition
    all_test = []
    for f in folds:
        tr, va, te = set(f["train_idx"]), set(f["val_idx"]), set(f["test_idx"])
        assert not (tr & va or tr & te or va & te), f"Overlap in fold {f['fold']}"
        all_test.extend(f["test_idx"])
    assert sorted(all_test) == list(indices), "Test folds don't partition the dataset"
    return folds

splits_binary = build_splits(aptos_df, task="binary", n_splits=cfg.n_splits, val_ratio=cfg.val_ratio)
splits_multiclass = build_splits(aptos_df, task="multiclass", n_splits=cfg.n_splits, val_ratio=cfg.val_ratio)

with open(ARTIFACT_ROOT / "splits" / "binary_folds.json", "w") as f:
    json.dump(splits_binary, f, indent=2)
with open(ARTIFACT_ROOT / "splits" / "multiclass_folds.json", "w") as f:
    json.dump(splits_multiclass, f, indent=2)

print("Splits saved.")
for task_name, splits in [("Binary", splits_binary), ("Multiclass", splits_multiclass)]:
    f0 = splits[0]
    print(f"  {task_name} fold 0: train={len(f0['train_idx'])}, val={len(f0['val_idx'])}, test={len(f0['test_idx'])}")
# ── Sensitivity analysis splits: binary with grade >= 1 threshold (M5) ──
splits_binary_alt = build_splits(aptos_df, task="binary_alt", n_splits=cfg.n_splits, val_ratio=cfg.val_ratio)
with open(ARTIFACT_ROOT / "splits" / "binary_alt_folds.json", "w") as f:
    json.dump(splits_binary_alt, f, indent=2)
print(f"  Binary-alt (grade>=1) fold 0: train={len(splits_binary_alt[0]['train_idx'])}, val={len(splits_binary_alt[0]['val_idx'])}, test={len(splits_binary_alt[0]['test_idx'])}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 11: MLP head + Linear head + training loop
# ═══════════════════════════════════════════════════════════
class EmbeddingArrayDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray):
        self.x = torch.from_numpy(x.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.int64))
    def __len__(self): return len(self.x)
    def __getitem__(self, idx): return self.x[idx], self.y[idx]


class MLPHead(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x): return self.net(x)


class LinearHead(nn.Module):                                    # ◄◄ NEW
    def __init__(self, in_dim: int, out_dim: int):               # ◄◄ NEW
        super().__init__()                                       # ◄◄ NEW
        self.fc = nn.Linear(in_dim, out_dim)                     # ◄◄ NEW
    def forward(self, x): return self.fc(x)                      # ◄◄ NEW


def make_class_weights(y_train: np.ndarray, num_classes: int) -> torch.Tensor:
    counts = np.bincount(y_train, minlength=num_classes).astype(np.float32)
    counts[counts == 0] = 1.0
    inv = 1.0 / counts
    w = inv / inv.sum() * num_classes
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


@torch.inference_mode()
def infer_logits(model: nn.Module, x: np.ndarray, batch_size: int = 1024) -> np.ndarray:
    model.eval()
    outs = []
    for s in range(0, len(x), batch_size):
        xb = torch.from_numpy(x[s:s+batch_size].astype(np.float32)).to(DEVICE)
        outs.append(model(xb).detach().cpu().numpy())
    return np.concatenate(outs, axis=0)


def train_fold_head(embeddings, labels, split, num_classes, seed, out_dir, run_tag, head_type="mlp"):  # ◄◄ NEW param
    set_global_seed(seed)
    out_dir.mkdir(parents=True, exist_ok=True)
    tr_idx = np.array(split["train_idx"], dtype=int)
    va_idx = np.array(split["val_idx"], dtype=int)
    te_idx = np.array(split["test_idx"], dtype=int)
    x_tr, y_tr = embeddings[tr_idx], labels[tr_idx]
    x_va, y_va = embeddings[va_idx], labels[va_idx]
    x_te, y_te = embeddings[te_idx], labels[te_idx]

    dl_tr = DataLoader(EmbeddingArrayDataset(x_tr, y_tr), batch_size=cfg.batch_size,
                       shuffle=True, num_workers=cfg.num_workers, pin_memory=True)

    # ── Head selection ──                                       # ◄◄ NEW block
    if head_type == "linear":                                    # ◄◄ NEW
        model = LinearHead(in_dim=embeddings.shape[1],           # ◄◄ NEW
                           out_dim=num_classes).to(DEVICE)       # ◄◄ NEW
    else:                                                        # ◄◄ NEW
        model = MLPHead(in_dim=embeddings.shape[1], hidden_dim=cfg.hidden_dim,
                        out_dim=num_classes, dropout=cfg.dropout).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=make_class_weights(y_tr, num_classes))
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.max_epochs)

    best_metric, bad_epochs, history = -1.0, 0, []
    ckpt_path = out_dir / f"{run_tag}_best.pt"

    for epoch in range(1, cfg.max_epochs + 1):
        model.train()
        losses = []
        for xb, yb in dl_tr:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))
        scheduler.step()

        va_logits = infer_logits(model, x_va)
        va_f1 = f1_score(y_va, va_logits.argmax(axis=1), average="macro")
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)),
                        "val_macro_f1": float(va_f1), "lr": float(optimizer.param_groups[0]["lr"])})
        if va_f1 > best_metric:
            best_metric, bad_epochs = va_f1, 0
            torch.save({
                "model_state_dict": model.state_dict(), "seed": seed, "run_tag": run_tag,
                "num_classes": num_classes, "in_dim": embeddings.shape[1],
                "hidden_dim": cfg.hidden_dim if head_type == "mlp" else 0,   # ◄◄ NEW
                "dropout": cfg.dropout if head_type == "mlp" else 0.0,       # ◄◄ NEW
                "head_type": head_type,                                      # ◄◄ NEW
                "best_val_macro_f1": best_metric,
            }, ckpt_path)
        else:
            bad_epochs += 1
            if bad_epochs >= cfg.patience:
                break

    best = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best["model_state_dict"])
    va_logits = infer_logits(model, x_va)
    te_logits = infer_logits(model, x_te)
    with open(out_dir / f"{run_tag}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    return {
        "checkpoint": ckpt_path, "history": history,
        "val": {"indices": va_idx, "targets": y_va, "logits": va_logits, "probs": softmax(va_logits, axis=1)},
        "test": {"indices": te_idx, "targets": y_te, "logits": te_logits, "probs": softmax(te_logits, axis=1)},
    }

print("Training APIs ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 12: Binary task CV run (APTOS, 3 models x 5 folds x 5 seeds)
# ═══════════════════════════════════════════════════════════
binary_manifest_rows = []
for model_key in ["medsiglip", "retfound", "efficientnet_b0"]:
    emb = np.load(EMBED_INDEX[model_key]["aptos"]["embeddings"])
    labels = aptos_df["binary_label"].values.astype(np.int64)
    for fold in splits_binary:
        fold_id = fold["fold"]
        for seed in cfg.seeds:
            run_tag = f"binary_{model_key}_fold{fold_id}_seed{seed}"
            run_dir = ARTIFACT_ROOT / "checkpoints" / "binary" / model_key / f"fold_{fold_id}" / f"seed_{seed}"
            pred_dir = ARTIFACT_ROOT / "predictions" / "binary" / model_key
            pred_dir.mkdir(parents=True, exist_ok=True)
            pred_npz = pred_dir / f"{run_tag}.npz"
            if pred_npz.exists():
                binary_manifest_rows.append({"task": "binary", "model": model_key, "fold": fold_id, "seed": seed,
                    "checkpoint": str(run_dir / f"{run_tag}_best.pt"), "prediction_npz": str(pred_npz)})
                continue
            out = train_fold_head(embeddings=emb, labels=labels, split=fold,
                                 num_classes=2, seed=seed, out_dir=run_dir, run_tag=run_tag)
            np.savez_compressed(pred_npz, model=model_key, task="binary", fold=fold_id, seed=seed,
                val_indices=out["val"]["indices"], val_targets=out["val"]["targets"],
                val_logits=out["val"]["logits"], val_probs=out["val"]["probs"],
                test_indices=out["test"]["indices"], test_targets=out["test"]["targets"],
                test_logits=out["test"]["logits"], test_probs=out["test"]["probs"])
            binary_manifest_rows.append({"task": "binary", "model": model_key, "fold": fold_id, "seed": seed,
                "checkpoint": str(out["checkpoint"]), "prediction_npz": str(pred_npz)})

binary_manifest = pd.DataFrame(binary_manifest_rows).sort_values(["model", "fold", "seed"]).reset_index(drop=True)
binary_manifest.to_csv(ARTIFACT_ROOT / "predictions" / "binary_manifest.csv", index=False)
print(f"Binary CV done: {binary_manifest.shape[0]} runs")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 12.5: Ablation (linear head) + Sensitivity (grade>=1 binary)
# ═══════════════════════════════════════════════════════════

# ── M4: Linear head ablation (3 models x 5 folds x 1 seed x 2 tasks) ──
ABLATION_SEED = 13
ablation_rows = []
for task_name, splits, label_col, n_cls in [
    ("binary", splits_binary, "binary_label", 2),
    ("multiclass", splits_multiclass, "grade", 5),
]:
    labels = aptos_df[label_col].values.astype(np.int64)
    for model_key in ["medsiglip", "retfound", "efficientnet_b0"]:
        emb = np.load(EMBED_INDEX[model_key]["aptos"]["embeddings"])
        for fold in splits:
            fold_id = fold["fold"]
            run_tag = f"linear_{task_name}_{model_key}_fold{fold_id}_seed{ABLATION_SEED}"
            run_dir = ARTIFACT_ROOT / "checkpoints" / "ablation_linear" / task_name / model_key / f"fold_{fold_id}"
            pred_dir = ARTIFACT_ROOT / "predictions" / "ablation_linear" / task_name / model_key
            pred_dir.mkdir(parents=True, exist_ok=True)
            pred_npz = pred_dir / f"{run_tag}.npz"
            if pred_npz.exists():
                ablation_rows.append({"task": task_name, "model": model_key, "fold": fold_id,
                    "head_type": "linear", "seed": ABLATION_SEED, "prediction_npz": str(pred_npz)})
                continue
            out = train_fold_head(embeddings=emb, labels=labels, split=fold,
                                 num_classes=n_cls, seed=ABLATION_SEED,
                                 out_dir=run_dir, run_tag=run_tag, head_type="linear")
            np.savez_compressed(pred_npz, model=model_key, task=task_name, fold=fold_id,
                seed=ABLATION_SEED, head_type="linear",
                test_indices=out["test"]["indices"], test_targets=out["test"]["targets"],
                test_logits=out["test"]["logits"], test_probs=out["test"]["probs"])
            ablation_rows.append({"task": task_name, "model": model_key, "fold": fold_id,
                "head_type": "linear", "seed": ABLATION_SEED, "prediction_npz": str(pred_npz)})

ablation_manifest = pd.DataFrame(ablation_rows)
ablation_manifest.to_csv(ARTIFACT_ROOT / "predictions" / "ablation_linear_manifest.csv", index=False)
print(f"Linear head ablation done: {len(ablation_manifest)} runs")


# ── M5: Sensitivity analysis — binary with grade >= 1 threshold ──
binary_alt_manifest_rows = []
labels_alt = aptos_df["binary_label_alt"].values.astype(np.int64)
for model_key in ["medsiglip", "retfound", "efficientnet_b0"]:
    emb = np.load(EMBED_INDEX[model_key]["aptos"]["embeddings"])
    for fold in splits_binary_alt:
        fold_id = fold["fold"]
        for seed in cfg.seeds:
            run_tag = f"binary_alt_{model_key}_fold{fold_id}_seed{seed}"
            run_dir = ARTIFACT_ROOT / "checkpoints" / "binary_alt" / model_key / f"fold_{fold_id}" / f"seed_{seed}"
            pred_dir = ARTIFACT_ROOT / "predictions" / "binary_alt" / model_key
            pred_dir.mkdir(parents=True, exist_ok=True)
            pred_npz = pred_dir / f"{run_tag}.npz"
            if pred_npz.exists():
                binary_alt_manifest_rows.append({"task": "binary_alt", "model": model_key,
                    "fold": fold_id, "seed": seed, "checkpoint": str(run_dir / f"{run_tag}_best.pt"),
                    "prediction_npz": str(pred_npz)})
                continue
            out = train_fold_head(embeddings=emb, labels=labels_alt, split=fold,
                                 num_classes=2, seed=seed, out_dir=run_dir, run_tag=run_tag)
            np.savez_compressed(pred_npz, model=model_key, task="binary_alt", fold=fold_id, seed=seed,
                val_indices=out["val"]["indices"], val_targets=out["val"]["targets"],
                val_logits=out["val"]["logits"], val_probs=out["val"]["probs"],
                test_indices=out["test"]["indices"], test_targets=out["test"]["targets"],
                test_logits=out["test"]["logits"], test_probs=out["test"]["probs"])
            binary_alt_manifest_rows.append({"task": "binary_alt", "model": model_key,
                "fold": fold_id, "seed": seed, "checkpoint": str(out["checkpoint"]),
                "prediction_npz": str(pred_npz)})

binary_alt_manifest = pd.DataFrame(binary_alt_manifest_rows).sort_values(["model", "fold", "seed"]).reset_index(drop=True)
binary_alt_manifest.to_csv(ARTIFACT_ROOT / "predictions" / "binary_alt_manifest.csv", index=False)
print(f"Binary-alt (grade>=1) CV done: {len(binary_alt_manifest)} runs")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 13: Multiclass (5-class) task CV run (APTOS, 3 models x 5 folds x 5 seeds)
# ═══════════════════════════════════════════════════════════
multiclass_manifest_rows = []
for model_key in ["medsiglip", "retfound", "efficientnet_b0"]:
    emb = np.load(EMBED_INDEX[model_key]["aptos"]["embeddings"])
    labels = aptos_df["grade"].values.astype(np.int64)
    for fold in splits_multiclass:
        fold_id = fold["fold"]
        for seed in cfg.seeds:
            run_tag = f"multiclass_{model_key}_fold{fold_id}_seed{seed}"
            run_dir = ARTIFACT_ROOT / "checkpoints" / "multiclass" / model_key / f"fold_{fold_id}" / f"seed_{seed}"
            pred_dir = ARTIFACT_ROOT / "predictions" / "multiclass" / model_key
            pred_dir.mkdir(parents=True, exist_ok=True)
            pred_npz = pred_dir / f"{run_tag}.npz"
            if pred_npz.exists():
                multiclass_manifest_rows.append({"task": "multiclass", "model": model_key, "fold": fold_id, "seed": seed,
                    "checkpoint": str(run_dir / f"{run_tag}_best.pt"), "prediction_npz": str(pred_npz)})
                continue
            out = train_fold_head(embeddings=emb, labels=labels, split=fold,
                                 num_classes=5, seed=seed, out_dir=run_dir, run_tag=run_tag)
            np.savez_compressed(pred_npz, model=model_key, task="multiclass", fold=fold_id, seed=seed,
                val_indices=out["val"]["indices"], val_targets=out["val"]["targets"],
                val_logits=out["val"]["logits"], val_probs=out["val"]["probs"],
                test_indices=out["test"]["indices"], test_targets=out["test"]["targets"],
                test_logits=out["test"]["logits"], test_probs=out["test"]["probs"])
            multiclass_manifest_rows.append({"task": "multiclass", "model": model_key, "fold": fold_id, "seed": seed,
                "checkpoint": str(out["checkpoint"]), "prediction_npz": str(pred_npz)})

multiclass_manifest = pd.DataFrame(multiclass_manifest_rows).sort_values(["model", "fold", "seed"]).reset_index(drop=True)
multiclass_manifest.to_csv(ARTIFACT_ROOT / "predictions" / "multiclass_manifest.csv", index=False)
print(f"Multiclass (5-class) CV done: {multiclass_manifest.shape[0]} runs")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 14: MESSIDOR-2 external inference (all fold heads, zero tuning)
# ═══════════════════════════════════════════════════════════
def load_head_from_checkpoint(ckpt_path: Path) -> nn.Module:
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    ht = ckpt.get("head_type", "mlp")
    if ht == "linear":
        model = LinearHead(in_dim=ckpt["in_dim"],
                           out_dim=ckpt["num_classes"]).to(DEVICE)
    else:
        model = MLPHead(in_dim=ckpt["in_dim"], hidden_dim=ckpt["hidden_dim"],
                        out_dim=ckpt["num_classes"], dropout=ckpt["dropout"]).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model

external_manifest_rows = []
for task_name, manifest in [("binary", binary_manifest),
                            ("multiclass", multiclass_manifest),
                            ("binary_alt", binary_alt_manifest)]:          # ◄◄ NEW
    for model_key, g_model in manifest.groupby("model"):
        ext_emb = np.load(EMBED_INDEX[model_key]["messidor"]["embeddings"])
        # ── Target selection per task ──                                   # ◄◄ CHANGED
        if task_name == "binary":                                           # ◄◄ CHANGED
            ext_targets = messidor_df["binary_label"].values                # ◄◄ CHANGED
        elif task_name == "binary_alt":                                     # ◄◄ NEW
            ext_targets = messidor_df["binary_label_alt"].values            # ◄◄ NEW
        else:                                                               # ◄◄ CHANGED
            ext_targets = messidor_df["grade"].values                       # ◄◄ CHANGED
        for _, row in g_model.iterrows():
            fold, seed = int(row["fold"]), int(row["seed"])
            ckpt_path = Path(row["checkpoint"])
            out_dir = ARTIFACT_ROOT / "predictions" / "external" / task_name / model_key
            out_dir.mkdir(parents=True, exist_ok=True)
            out_npz = out_dir / f"{task_name}_{model_key}_fold{fold}_seed{seed}.npz"
            if out_npz.exists():
                external_manifest_rows.append({"task": task_name, "model": model_key, "fold": fold, "seed": seed, "external_npz": str(out_npz)})
                continue
            head = load_head_from_checkpoint(ckpt_path)
            logits = infer_logits(head, ext_emb)
            probs = softmax(logits, axis=1)
            np.savez_compressed(out_npz, task=task_name, model=model_key, fold=fold, seed=seed,
                                indices=np.arange(len(ext_targets)), targets=ext_targets, logits=logits, probs=probs)
            external_manifest_rows.append({"task": task_name, "model": model_key, "fold": fold, "seed": seed, "external_npz": str(out_npz)})

external_manifest = pd.DataFrame(external_manifest_rows).sort_values(["task", "model", "fold", "seed"]).reset_index(drop=True)
external_manifest.to_csv(ARTIFACT_ROOT / "predictions" / "external_manifest.csv", index=False)

# ── Seed-average + fold-ensemble ──
ensemble_rows = []
for (task_name, model_key, fold), grp in external_manifest.groupby(["task", "model", "fold"]):
    arrs = [np.load(p) for p in grp["external_npz"].tolist()]
    targets = arrs[0]["targets"]
    probs = np.mean([a["probs"] for a in arrs], axis=0)
    logits = np.mean([a["logits"] for a in arrs], axis=0)
    out_dir = ARTIFACT_ROOT / "predictions" / "external" / task_name / model_key
    seedavg_npz = out_dir / f"{task_name}_{model_key}_fold{fold}_seedavg.npz"
    np.savez_compressed(seedavg_npz, task=task_name, model=model_key, fold=fold, targets=targets, logits=logits, probs=probs)
    ensemble_rows.append({"task": task_name, "model": model_key, "fold": fold, "kind": "seedavg", "npz": str(seedavg_npz)})

for (task_name, model_key), grp in pd.DataFrame(ensemble_rows).groupby(["task", "model"]):
    arrs = [np.load(p) for p in grp["npz"].tolist()]
    targets = arrs[0]["targets"]
    probs = np.mean([a["probs"] for a in arrs], axis=0)
    logits = np.mean([a["logits"] for a in arrs], axis=0)
    out_dir = ARTIFACT_ROOT / "predictions" / "external" / task_name / model_key
    ens_npz = out_dir / f"{task_name}_{model_key}_foldensemble.npz"
    np.savez_compressed(ens_npz, task=task_name, model=model_key, targets=targets, logits=logits, probs=probs)
    ensemble_rows.append({"task": task_name, "model": model_key, "fold": -1, "kind": "foldensemble", "npz": str(ens_npz)})

pd.DataFrame(ensemble_rows).to_csv(ARTIFACT_ROOT / "predictions" / "external_ensemble_manifest.csv", index=False)
print(f"External inference complete: {len(external_manifest)} runs + ensembles")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 15: Temperature scaling (fit on val only, apply to test + external)
# ═══════════════════════════════════════════════════════════
def apply_temperature(logits: np.ndarray, T: float) -> np.ndarray:
    return logits / float(T)

def nll_from_logits(logits: np.ndarray, y_true: np.ndarray) -> float:
    probs = softmax(logits, axis=1)
    ll = np.log(np.clip(probs[np.arange(len(y_true)), y_true], 1e-12, 1.0))
    return float(-ll.mean())

def fit_temperature(logits: np.ndarray, y_true: np.ndarray) -> float:
    def obj(t):
        return nll_from_logits(logits / float(np.clip(t, 1e-3, 100.0)), y_true)
    res = minimize_scalar(obj, bounds=(0.05, 10.0), method="bounded")
    return float(np.clip(res.x, 1e-3, 100.0)) if res.success else 1.0

ts_rows = []
for task_name, manifest in [("binary", binary_manifest), ("multiclass", multiclass_manifest), ("binary_alt", binary_alt_manifest)]:  # ◄◄ NEW
    ext_sub = external_manifest[external_manifest["task"] == task_name]
    for _, row in manifest.iterrows():
        model_key, fold, seed = row["model"], int(row["fold"]), int(row["seed"])
        dev_npz = Path(row["prediction_npz"])
        ext_match = ext_sub[(ext_sub["model"] == model_key) & (ext_sub["fold"] == fold) & (ext_sub["seed"] == seed)]
        if ext_match.empty:
            continue
        ext_npz = Path(ext_match["external_npz"].iloc[0])
        dev, ext = np.load(dev_npz), np.load(ext_npz)
        T = fit_temperature(dev["val_logits"], dev["val_targets"])

        val_pre, test_pre, ext_pre = dev["val_logits"], dev["test_logits"], ext["logits"]
        val_post = apply_temperature(val_pre, T)
        test_post = apply_temperature(test_pre, T)
        ext_post = apply_temperature(ext_pre, T)

        out_dir = ARTIFACT_ROOT / "predictions" / "ts" / task_name / model_key
        out_dir.mkdir(parents=True, exist_ok=True)
        out_npz = out_dir / f"{task_name}_{model_key}_fold{fold}_seed{seed}_ts.npz"
        np.savez_compressed(out_npz,
            task=task_name, model=model_key, fold=fold, seed=seed, temperature=T,
            val_indices=dev["val_indices"], val_targets=dev["val_targets"],
            val_logits_pre=val_pre, val_logits_post=val_post,
            val_probs_pre=softmax(val_pre, axis=1), val_probs_post=softmax(val_post, axis=1),
            test_indices=dev["test_indices"], test_targets=dev["test_targets"],
            test_logits_pre=test_pre, test_logits_post=test_post,
            test_probs_pre=softmax(test_pre, axis=1), test_probs_post=softmax(test_post, axis=1),
            ext_indices=ext["indices"], ext_targets=ext["targets"],
            ext_logits_pre=ext_pre, ext_logits_post=ext_post,
            ext_probs_pre=softmax(ext_pre, axis=1), ext_probs_post=softmax(ext_post, axis=1),
        )
        ts_rows.append({"task": task_name, "model": model_key, "fold": fold, "seed": seed,
                         "temperature": T, "ts_npz": str(out_npz)})

ts_manifest = pd.DataFrame(ts_rows).sort_values(["task", "model", "fold", "seed"]).reset_index(drop=True)
ts_manifest.to_csv(ARTIFACT_ROOT / "predictions" / "ts_manifest.csv", index=False)
print(f"Temperature scaling complete: {len(ts_manifest)} runs")
print(f"Mean T: {ts_manifest['temperature'].mean():.3f} ± {ts_manifest['temperature'].std():.3f}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 16: Metric computation (AUC, F1, ECE, Brier, QWK)
# ═══════════════════════════════════════════════════════════
def expected_calibration_error(y_true, probs, n_bins=10):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    corr = (pred == y_true).astype(np.float32)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece, n = 0.0, len(y_true)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(corr[mask].mean() - conf[mask].mean())
    return float(ece)


def compute_metrics(task, y_true, probs, ece_bins=10):
    pred = probs.argmax(axis=1)
    out = {}
    if task in ("binary", "binary_alt"):    # ◄◄ CHANGED
        prob_pos = probs[:, 1]
        out["auc"] = float(roc_auc_score(y_true, prob_pos))
        out["f1"] = float(f1_score(y_true, pred, average="binary"))
        out["ece"] = expected_calibration_error(y_true, probs, n_bins=ece_bins)
        out["brier"] = float(brier_score_loss(y_true, prob_pos))
    elif task == "multiclass":
        out["macro_f1"] = float(f1_score(y_true, pred, average="macro"))
        out["qwk"] = float(cohen_kappa_score(y_true, pred, weights="quadratic"))
        out["balanced_acc"] = float(balanced_accuracy_score(y_true, pred))
        out["ece"] = expected_calibration_error(y_true, probs, n_bins=ece_bins)
        # ── M3: multiclass Brier score (was missing) ──
        one_hot = np.eye(probs.shape[1])[y_true]
        out["brier"] = float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))
    return out


# ── Aggregate: per seed-fold, per fold (seed-averaged), per model (fold-averaged) ──
seed_fold_rows, fold_rows = [], []
for task_name in ["binary", "multiclass", "binary_alt"]:    # ◄◄ NEW
    sub = ts_manifest[ts_manifest["task"] == task_name]
    for (model_key, fold), g in sub.groupby(["model", "fold"]):
        seed_probs_dev_pre, seed_probs_dev_post = [], []
        seed_probs_ext_pre, seed_probs_ext_post = [], []
        targets_dev, targets_ext, test_idx_ref = None, None, None
        for _, row in g.iterrows():
            arr = np.load(row["ts_npz"])
            y_dev, y_ext = arr["test_targets"], arr["ext_targets"]
            idx_dev = arr["test_indices"]
            if targets_dev is None:
                targets_dev, targets_ext, test_idx_ref = y_dev, y_ext, idx_dev
            else:
                assert np.array_equal(targets_dev, y_dev) and np.array_equal(targets_ext, y_ext)

            # Per seed-fold metrics
            dev_pre = compute_metrics(task_name, y_dev, arr["test_probs_pre"], ece_bins=cfg.ece_bins)
            dev_post = compute_metrics(task_name, y_dev, arr["test_probs_post"], ece_bins=cfg.ece_bins)
            ext_pre = compute_metrics(task_name, y_ext, arr["ext_probs_pre"], ece_bins=cfg.ece_bins)
            ext_post = compute_metrics(task_name, y_ext, arr["ext_probs_post"], ece_bins=cfg.ece_bins)
            base = {"task": task_name, "model": model_key, "fold": fold, "seed": int(row["seed"])}
            for k, v in dev_pre.items(): base[f"dev_pre_{k}"] = v
            for k, v in dev_post.items(): base[f"dev_post_{k}"] = v
            for k, v in ext_pre.items(): base[f"ext_pre_{k}"] = v
            for k, v in ext_post.items(): base[f"ext_post_{k}"] = v
            seed_fold_rows.append(base)
            seed_probs_dev_pre.append(arr["test_probs_pre"]); seed_probs_dev_post.append(arr["test_probs_post"])
            seed_probs_ext_pre.append(arr["ext_probs_pre"]); seed_probs_ext_post.append(arr["ext_probs_post"])

        # Seed-averaged fold metrics
        dev_pre_avg = np.mean(seed_probs_dev_pre, axis=0)
        dev_post_avg = np.mean(seed_probs_dev_post, axis=0)
        ext_pre_avg = np.mean(seed_probs_ext_pre, axis=0)
        ext_post_avg = np.mean(seed_probs_ext_post, axis=0)
        fr = {"task": task_name, "model": model_key, "fold": fold}
        for k, v in compute_metrics(task_name, targets_dev, dev_pre_avg, ece_bins=cfg.ece_bins).items(): fr[f"dev_pre_{k}"] = v
        for k, v in compute_metrics(task_name, targets_dev, dev_post_avg, ece_bins=cfg.ece_bins).items(): fr[f"dev_post_{k}"] = v
        for k, v in compute_metrics(task_name, targets_ext, ext_pre_avg, ece_bins=cfg.ece_bins).items(): fr[f"ext_pre_{k}"] = v
        for k, v in compute_metrics(task_name, targets_ext, ext_post_avg, ece_bins=cfg.ece_bins).items(): fr[f"ext_post_{k}"] = v
        fold_rows.append(fr)

        # Save fold-level seed-averaged predictions
        out_dir = ARTIFACT_ROOT / "metrics" / "fold_seedavg_preds" / task_name / model_key
        out_dir.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(out_dir / f"fold_{fold}.npz",
            test_indices=test_idx_ref, test_targets=targets_dev,
            test_probs_pre=dev_pre_avg, test_probs_post=dev_post_avg,
            ext_targets=targets_ext, ext_probs_pre=ext_pre_avg, ext_probs_post=ext_post_avg)

pd.DataFrame(seed_fold_rows).to_csv(ARTIFACT_ROOT / "metrics" / "seed_fold_metrics.csv", index=False)
fold_metrics = pd.DataFrame(fold_rows)
fold_metrics.to_csv(ARTIFACT_ROOT / "metrics" / "fold_metrics.csv", index=False)

# Model-level summary (mean ± SD across folds)
summary_rows = []
for (task_name, model_key), g in fold_metrics.groupby(["task", "model"]):
    numeric_cols = [c for c in g.columns if c.startswith(("dev_", "ext_"))]
    means, sds = g[numeric_cols].mean(), g[numeric_cols].std(ddof=1)
    row = {"task": task_name, "model": model_key}
    for c in numeric_cols:
        row[f"{c}_mean"] = float(means[c]); row[f"{c}_sd"] = float(sds[c])
    summary_rows.append(row)
pd.DataFrame(summary_rows).to_csv(ARTIFACT_ROOT / "metrics" / "summary_metrics.csv", index=False)

# OOF concatenation for statistical tests
for task_name in ["binary", "multiclass", "binary_alt"]:    # ◄◄ NEW
    for model_key in ["medsiglip", "retfound", "efficientnet_b0"]:
        fold_dir = ARTIFACT_ROOT / "metrics" / "fold_seedavg_preds" / task_name / model_key
        fold_files = sorted(fold_dir.glob("fold_*.npz"))
        dev_idx, dev_y, dev_pre, dev_post = [], [], [], []
        ext_pre_stack, ext_post_stack, ext_y_ref = [], [], None
        for fp in fold_files:
            arr = np.load(fp)
            dev_idx.append(arr["test_indices"]); dev_y.append(arr["test_targets"])
            dev_pre.append(arr["test_probs_pre"]); dev_post.append(arr["test_probs_post"])
            ext_pre_stack.append(arr["ext_probs_pre"]); ext_post_stack.append(arr["ext_probs_post"])
            if ext_y_ref is None: ext_y_ref = arr["ext_targets"]
        dev_idx = np.concatenate(dev_idx); dev_y = np.concatenate(dev_y)
        dev_pre = np.concatenate(dev_pre); dev_post = np.concatenate(dev_post)
        order = np.argsort(dev_idx)
        np.savez_compressed(ARTIFACT_ROOT / "metrics" / f"stats_input_{task_name}_{model_key}.npz",
            dev_indices=dev_idx[order], dev_targets=dev_y[order],
            dev_probs_pre=dev_pre[order], dev_probs_post=dev_post[order],
            ext_targets=ext_y_ref,
            ext_probs_pre=np.mean(ext_pre_stack, axis=0),
            ext_probs_post=np.mean(ext_post_stack, axis=0))

# Per-grade metrics
per_grade_rows = []
for model_key in ["medsiglip", "retfound", "efficientnet_b0"]:
    arr = np.load(ARTIFACT_ROOT / "metrics" / f"stats_input_multiclass_{model_key}.npz")
    for ds, y, probs in [("dev", arr["dev_targets"], arr["dev_probs_post"]),
                         ("external", arr["ext_targets"], arr["ext_probs_post"])]:
        pred = probs.argmax(axis=1)
        p, r, f, s = precision_recall_fscore_support(y, pred, labels=[0,1,2,3,4], zero_division=0)
        for g in range(5):
            per_grade_rows.append({"model": model_key, "dataset": ds, "grade": g,
                                   "precision": float(p[g]), "recall": float(r[g]),
                                   "f1": float(f[g]), "support": int(s[g])})
pd.DataFrame(per_grade_rows).to_csv(ARTIFACT_ROOT / "metrics" / "per_grade_metrics.csv", index=False)
# ── Ablation (linear head) summary metrics ──
ablation_manifest = pd.read_csv(ARTIFACT_ROOT / "predictions" / "ablation_linear_manifest.csv")
ablation_metric_rows = []
for (task_name, model_key), grp in ablation_manifest.groupby(["task", "model"]):
    fold_metrics_list = []
    for _, row in grp.iterrows():
        arr = np.load(row["prediction_npz"])
        m = compute_metrics(task_name, arr["test_targets"], arr["test_probs"], ece_bins=cfg.ece_bins)
        fold_metrics_list.append(m)
    keys = fold_metrics_list[0].keys()
    summary = {}
    for k in keys:
        vals = [fm[k] for fm in fold_metrics_list]
        summary[f"{k}_mean"] = float(np.mean(vals))
        summary[f"{k}_sd"] = float(np.std(vals, ddof=1))
    summary.update({"task": task_name, "model": model_key, "head_type": "linear"})
    ablation_metric_rows.append(summary)
pd.DataFrame(ablation_metric_rows).to_csv(ARTIFACT_ROOT / "metrics" / "ablation_linear_summary.csv", index=False)
print("Ablation (linear head) metrics saved.")
print("All metrics computed and saved.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 17: Statistical tests (DeLong, paired bootstrap, McNemar, BH-FDR)
#          + cluster-robust bootstrap for external set (M2)
#          + multiclass ECE/Brier bootstrap (M3)
# ═══════════════════════════════════════════════════════════

# ── DeLong test (Sun & Xu midrank) ──
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N, dtype=float); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1); i = j
    T2 = np.empty(N, dtype=float); T2[J] = T + 1
    return T2

def fast_delong(predictions_sorted_transposed, label_1_count):
    m = label_1_count; n = predictions_sorted_transposed.shape[1] - m
    pos = predictions_sorted_transposed[:, :m]; neg = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]
    tx, ty, tz = np.empty([k, m]), np.empty([k, n]), np.empty([k, m + n])
    for r in range(k):
        tx[r] = compute_midrank(pos[r]); ty[r] = compute_midrank(neg[r])
        tz[r] = compute_midrank(predictions_sorted_transposed[r])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    return aucs, np.cov(v01) / m + np.cov(v10) / n

def delong_roc_test(y_true, pred1, pred2):
    order = np.argsort(-y_true)
    preds = np.vstack((pred1, pred2))[:, order]
    aucs, cov = fast_delong(preds, int(y_true.sum()))
    diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = np.abs(diff) / np.sqrt(max(var, 1e-12))
    p = 2 * (1 - stats.norm.cdf(z))
    return float(diff), float(p), float(var)

# ── Paired bootstrap (image-level, used for dev set) ──
def bootstrap_delta(y_true, pred_a, pred_b, metric_fn, n_iter=1000, seed=42):
    rng = np.random.default_rng(seed); n = len(y_true); diffs = []
    for _ in range(n_iter):
        idx = rng.integers(0, n, size=n)
        diffs.append(metric_fn(y_true[idx], pred_a[idx]) - metric_fn(y_true[idx], pred_b[idx]))
    diffs = np.array(diffs)
    delta = float(np.mean(diffs))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = 2 * min(float((diffs <= 0).mean()), float((diffs >= 0).mean()))
    return delta, float(lo), float(hi), max(p, 1.0 / n_iter)

# ── Cluster-robust bootstrap (M2: patient-level resampling for MESSIDOR-2) ──  # ◄◄ NEW
_messidor_patient_ids = messidor_df["patient_id"].values                        # ◄◄ NEW

def cluster_bootstrap_delta(y_true, pred_a, pred_b, patient_ids,                # ◄◄ NEW
                            metric_fn, n_iter=2000, seed=42):                   # ◄◄ NEW
    rng = np.random.default_rng(seed)                                           # ◄◄ NEW
    unique_patients = np.unique(patient_ids)                                    # ◄◄ NEW
    n_patients = len(unique_patients)                                           # ◄◄ NEW
    pat2idx = {p: np.where(patient_ids == p)[0] for p in unique_patients}       # ◄◄ NEW
    diffs = []                                                                  # ◄◄ NEW
    for _ in range(n_iter):                                                     # ◄◄ NEW
        sampled = rng.choice(unique_patients, size=n_patients, replace=True)    # ◄◄ NEW
        idx = np.concatenate([pat2idx[p] for p in sampled])                     # ◄◄ NEW
        diffs.append(metric_fn(y_true[idx], pred_a[idx]) -                      # ◄◄ NEW
                     metric_fn(y_true[idx], pred_b[idx]))                       # ◄◄ NEW
    diffs = np.array(diffs)                                                     # ◄◄ NEW
    delta = float(np.mean(diffs))                                               # ◄◄ NEW
    lo, hi = np.percentile(diffs, [2.5, 97.5])                                 # ◄◄ NEW
    p = 2 * min(float((diffs <= 0).mean()), float((diffs >= 0).mean()))         # ◄◄ NEW
    return delta, float(lo), float(hi), max(p, 1.0 / n_iter)                   # ◄◄ NEW

# ── Metric functions for bootstrap ──
def metric_binary_auc(y, probs): return float(roc_auc_score(y, probs[:, 1]))    # ◄◄ NEW
def metric_binary_brier(y, probs): return float(brier_score_loss(y, probs[:, 1]))
def metric_binary_ece(y, probs): return expected_calibration_error(y, probs, n_bins=cfg.ece_bins)
def metric_mc_macro_f1(y, probs): return float(f1_score(y, probs.argmax(1), average="macro"))
def metric_mc_qwk(y, probs): return float(cohen_kappa_score(y, probs.argmax(1), weights="quadratic"))
def metric_mc_ece(y, probs): return expected_calibration_error(y, probs, n_bins=cfg.ece_bins)  # ◄◄ NEW
def metric_mc_brier(y, probs):                                                                 # ◄◄ NEW
    one_hot = np.eye(probs.shape[1])[y]                                                        # ◄◄ NEW
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))                              # ◄◄ NEW

# ── Helper: choose bootstrap function based on dataset ──                      # ◄◄ NEW
def _bootstrap(dataset_name, y, pa, pb, metric_fn):                             # ◄◄ NEW
    if dataset_name == "ext":                                                   # ◄◄ NEW
        return cluster_bootstrap_delta(y, pa, pb,                               # ◄◄ NEW
                   patient_ids=_messidor_patient_ids,                           # ◄◄ NEW
                   metric_fn=metric_fn, n_iter=2000, seed=cfg.split_seed)       # ◄◄ NEW
    else:                                                                       # ◄◄ NEW
        return bootstrap_delta(y, pa, pb,                                       # ◄◄ NEW
                   metric_fn=metric_fn, n_iter=cfg.bootstrap_iters,             # ◄◄ NEW
                   seed=cfg.split_seed)                                         # ◄◄ NEW

# ── Run all pairwise tests ──
pairs = [("medsiglip", "retfound"), ("medsiglip", "efficientnet_b0"), ("retfound", "efficientnet_b0")]
stat_rows = []

for dataset_name in ["dev", "ext"]:
    bin_data = {m: np.load(ARTIFACT_ROOT / "metrics" / f"stats_input_binary_{m}.npz") for m in ["medsiglip", "retfound", "efficientnet_b0"]}
    mul_data = {m: np.load(ARTIFACT_ROOT / "metrics" / f"stats_input_multiclass_{m}.npz") for m in ["medsiglip", "retfound", "efficientnet_b0"]}
    yb = bin_data["medsiglip"][f"{dataset_name}_targets"].astype(int)
    ym = mul_data["medsiglip"][f"{dataset_name}_targets"].astype(int)

    for a, b in pairs:
        pa = bin_data[a][f"{dataset_name}_probs_post"][:, 1].astype(float)
        pb = bin_data[b][f"{dataset_name}_probs_post"][:, 1].astype(float)

        # DeLong (binary AUC) — unadjusted for clustering on ext
        d_auc, p_del, _ = delong_roc_test(yb, pa, pb)
        test_label = "delong_unadj" if dataset_name == "ext" else "delong"      # ◄◄ NEW
        stat_rows.append({"dataset": dataset_name, "task": "binary", "metric": "auc", "test": test_label,
                          "model_a": a, "model_b": b, "delta": d_auc, "ci_low": np.nan, "ci_high": np.nan, "p_value": p_del})

        # Cluster-robust bootstrap AUC (ext only)                               # ◄◄ NEW block
        if dataset_name == "ext":                                                # ◄◄ NEW
            d, lo, hi, p = cluster_bootstrap_delta(yb,                           # ◄◄ NEW
                bin_data[a]["ext_probs_post"], bin_data[b]["ext_probs_post"],     # ◄◄ NEW
                patient_ids=_messidor_patient_ids,                               # ◄◄ NEW
                metric_fn=metric_binary_auc, n_iter=2000, seed=cfg.split_seed)   # ◄◄ NEW
            stat_rows.append({"dataset": "ext", "task": "binary", "metric": "auc",  # ◄◄ NEW
                "test": "cluster_bootstrap",                                     # ◄◄ NEW
                "model_a": a, "model_b": b, "delta": d,                         # ◄◄ NEW
                "ci_low": lo, "ci_high": hi, "p_value": p})                     # ◄◄ NEW

        # McNemar (binary correctness)
        pred_a = (pa >= 0.5).astype(int)
        pred_b = (pb >= 0.5).astype(int)
        corr_a = (pred_a == yb).astype(int)
        corr_b = (pred_b == yb).astype(int)
        n11 = int(np.sum((corr_a == 1) & (corr_b == 1)))
        n10 = int(np.sum((corr_a == 1) & (corr_b == 0)))
        n01 = int(np.sum((corr_a == 0) & (corr_b == 1)))
        n00 = int(np.sum((corr_a == 0) & (corr_b == 0)))
        table = np.array([[n11, n10], [n01, n00]])
        if n10 + n01 < 25:
            p_mc = float(mcnemar(table, exact=True).pvalue)
        else:
            p_mc = float(mcnemar(table, exact=False, correction=True).pvalue)
        stat_rows.append({"dataset": dataset_name, "task": "binary", "metric": "correctness", "test": "mcnemar",
                          "model_a": a, "model_b": b, "delta": float((corr_a - corr_b).mean()),
                          "ci_low": np.nan, "ci_high": np.nan, "p_value": p_mc})

        # Bootstrap: ECE, Brier (binary) — cluster on ext, image-level on dev   # ◄◄ CHANGED
        for mname, fn in [("ece", metric_binary_ece), ("brier", metric_binary_brier)]:
            d, lo, hi, p = _bootstrap(dataset_name, yb,                          # ◄◄ CHANGED
                bin_data[a][f"{dataset_name}_probs_post"],                       # ◄◄ CHANGED
                bin_data[b][f"{dataset_name}_probs_post"], fn)                   # ◄◄ CHANGED
            stat_rows.append({"dataset": dataset_name, "task": "binary", "metric": mname,
                              "test": "cluster_bootstrap" if dataset_name == "ext" else "bootstrap",  # ◄◄ NEW
                              "model_a": a, "model_b": b, "delta": d, "ci_low": lo, "ci_high": hi, "p_value": p})

        # Bootstrap: macro-F1, QWK, ECE, Brier (multiclass) — ECE+Brier new (M3)  # ◄◄ CHANGED
        for mname, fn in [("macro_f1", metric_mc_macro_f1), ("qwk", metric_mc_qwk),
                          ("ece", metric_mc_ece), ("brier", metric_mc_brier)]:   # ◄◄ NEW: ece, brier added
            d, lo, hi, p = _bootstrap(dataset_name, ym,                          # ◄◄ CHANGED
                mul_data[a][f"{dataset_name}_probs_post"],                       # ◄◄ CHANGED
                mul_data[b][f"{dataset_name}_probs_post"], fn)                   # ◄◄ CHANGED
            stat_rows.append({"dataset": dataset_name, "task": "multiclass", "metric": mname,
                              "test": "cluster_bootstrap" if dataset_name == "ext" else "bootstrap",  # ◄◄ NEW
                              "model_a": a, "model_b": b, "delta": d, "ci_low": lo, "ci_high": hi, "p_value": p})

stats_df = pd.DataFrame(stat_rows)
stats_df["p_bh"] = np.nan
stats_df["significant_bh"] = False
for _, g in stats_df.groupby(["dataset", "task", "test"]):
    if len(g) > 1:
        rej, p_adj, _, _ = multipletests(g["p_value"].values, alpha=0.05, method="fdr_bh")
        stats_df.loc[g.index, "p_bh"] = p_adj
        stats_df.loc[g.index, "significant_bh"] = rej
    else:
        stats_df.loc[g.index, "p_bh"] = g["p_value"].values
stats_df.to_csv(ARTIFACT_ROOT / "metrics" / "pairwise_stats.csv", index=False)
print("Statistical tests done.")
print(stats_df[["dataset","task","metric","test","model_a","model_b","delta","p_value","p_bh"]].to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 18: Table 1 — Dataset summary + contamination audit (NEW)
# ═══════════════════════════════════════════════════════════

# ── Panel A: Dataset summary ──
panel_a_rows = []
for ds_name, df, role in [("APTOS 2019", aptos_df, "Development (5-fold CV)"),
                           ("MESSIDOR-2", messidor_df, "External test (zero tuning)")]:
    n = len(df)
    dist = df["grade"].value_counts().sort_index()
    binary = df["binary_label"].value_counts().sort_index()
    panel_a_rows.append({
        "Dataset": ds_name, "Role": role, "N": n,
        "Grade 0": int(dist.get(0, 0)), "Grade 1": int(dist.get(1, 0)),
        "Grade 2": int(dist.get(2, 0)), "Grade 3": int(dist.get(3, 0)),
        "Grade 4": int(dist.get(4, 0)),
        "Non-Ref (0-1)": int(binary.get(0, 0)), "Referable (2-4)": int(binary.get(1, 0)),
    })
table1a = pd.DataFrame(panel_a_rows)

# ── Panel B: Contamination audit ──
panel_b_rows = [
    {"Dataset": "APTOS 2019", "Role": "Development", "MedSigLIP": "Not confirmed", "RETFound": "Eval only", "EfficientNet-B0": "No"},
    {"Dataset": "MESSIDOR-2", "Role": "External", "MedSigLIP": "Not confirmed", "RETFound": "Eval only", "EfficientNet-B0": "No"},
    {"Dataset": "EyePACS", "Role": "EXCLUDED", "MedSigLIP": "Pretraining", "RETFound": "Pretraining", "EfficientNet-B0": "No"},
    {"Dataset": "ImageNet-1K", "Role": "N/A", "MedSigLIP": "No", "RETFound": "No", "EfficientNet-B0": "Pretraining"},
]
table1b = pd.DataFrame(panel_b_rows)

# ── Save ──
for fmt_name, tbl, fname in [("Panel A", table1a, "table1a_dataset_summary"),
                               ("Panel B", table1b, "table1b_contamination_audit")]:
    tbl.to_csv(ARTIFACT_ROOT / "tables" / f"{fname}.csv", index=False)
    with open(ARTIFACT_ROOT / "tables" / f"{fname}.md", "w") as f:
        f.write(tbl.to_markdown(index=False))
    with open(ARTIFACT_ROOT / "tables" / f"{fname}.tex", "w") as f:
        f.write(tbl.to_latex(index=False, escape=True))
    print(f"\nTable 1 {fmt_name}:")
    print(tbl.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 19: Table 2 — Main performance (Dev + External, pre/post TS)
# ═══════════════════════════════════════════════════════════
summary = pd.read_csv(ARTIFACT_ROOT / "metrics" / "summary_metrics.csv")

def fmt(mean_val, sd_val, digits=3):
    return f"{mean_val:.{digits}f} ± {sd_val:.{digits}f}"

rows = []
for model_key in ["medsiglip", "retfound", "efficientnet_b0"]:
    b = summary[(summary["task"] == "binary") & (summary["model"] == model_key)].iloc[0]
    m = summary[(summary["task"] == "multiclass") & (summary["model"] == model_key)].iloc[0]
    rows.append({
        "Model": model_key,
        "Dev AUC": fmt(b["dev_post_auc_mean"], b["dev_post_auc_sd"]),
        "Ext AUC": fmt(b["ext_post_auc_mean"], b["ext_post_auc_sd"]),
        "Dev Macro-F1": fmt(m["dev_post_macro_f1_mean"], m["dev_post_macro_f1_sd"]),
        "Ext Macro-F1": fmt(m["ext_post_macro_f1_mean"], m["ext_post_macro_f1_sd"]),
        "Dev ECE pre": fmt(b["dev_pre_ece_mean"], b["dev_pre_ece_sd"]),
        "Dev ECE post": fmt(b["dev_post_ece_mean"], b["dev_post_ece_sd"]),
        "Ext ECE pre": fmt(b["ext_pre_ece_mean"], b["ext_pre_ece_sd"]),
        "Ext ECE post": fmt(b["ext_post_ece_mean"], b["ext_post_ece_sd"]),
        "Dev Brier": fmt(b["dev_post_brier_mean"], b["dev_post_brier_sd"]),
        "Ext Brier": fmt(b["ext_post_brier_mean"], b["ext_post_brier_sd"]),
        "Dev QWK": fmt(m["dev_post_qwk_mean"], m["dev_post_qwk_sd"]),
        "Ext QWK": fmt(m["ext_post_qwk_mean"], m["ext_post_qwk_sd"]),
    })

table2 = pd.DataFrame(rows)
for ext in ["csv", "md", "tex"]:
    path = ARTIFACT_ROOT / "tables" / f"table2_main_performance.{ext}"
    if ext == "csv": table2.to_csv(path, index=False)
    elif ext == "md":
        with open(path, "w") as f: f.write(table2.to_markdown(index=False))
    elif ext == "tex":
        with open(path, "w") as f: f.write(table2.to_latex(index=False, escape=False))
print("Table 2 — Main Performance:")
print(table2.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 20: Table 3 — Pairwise statistical comparisons
# ═══════════════════════════════════════════════════════════
stats_df = pd.read_csv(ARTIFACT_ROOT / "metrics" / "pairwise_stats.csv")

rows = []
for dataset_name in ["dev", "ext"]:
    for a, b in [("medsiglip", "retfound"), ("medsiglip", "efficientnet_b0"), ("retfound", "efficientnet_b0")]:
        d = stats_df[(stats_df["dataset"]==dataset_name) & (stats_df["task"]=="binary") & (stats_df["metric"]=="auc") & (stats_df["test"]=="delong") & (stats_df["model_a"]==a) & (stats_df["model_b"]==b)]
        m = stats_df[(stats_df["dataset"]==dataset_name) & (stats_df["task"]=="multiclass") & (stats_df["metric"]=="macro_f1") & (stats_df["test"]=="bootstrap") & (stats_df["model_a"]==a) & (stats_df["model_b"]==b)]
        c = stats_df[(stats_df["dataset"]==dataset_name) & (stats_df["task"]=="binary") & (stats_df["test"]=="mcnemar") & (stats_df["model_a"]==a) & (stats_df["model_b"]==b)]
        if d.empty or m.empty or c.empty: continue
        d, m, c = d.iloc[0], m.iloc[0], c.iloc[0]
        rows.append({
            "Dataset": dataset_name, "Pair": f"{a} vs {b}",
            "ΔAUC": f"{d['delta']:.4f}", "DeLong p": f"{d['p_value']:.4f}", "DeLong p(BH)": f"{d['p_bh']:.4f}",
            "ΔMacro-F1": f"{m['delta']:.4f}", "F1 95%CI": f"[{m['ci_low']:.4f},{m['ci_high']:.4f}]",
            "Bootstrap p(BH)": f"{m['p_bh']:.4f}",
            "McNemar p": f"{c['p_value']:.4f}", "McNemar p(BH)": f"{c['p_bh']:.4f}",
        })

table3 = pd.DataFrame(rows)
for ext in ["csv", "md", "tex"]:
    path = ARTIFACT_ROOT / "tables" / f"table3_pairwise_stats.{ext}"
    if ext == "csv": table3.to_csv(path, index=False)
    elif ext == "md":
        with open(path, "w") as f: f.write(table3.to_markdown(index=False))
    elif ext == "tex":
        with open(path, "w") as f: f.write(table3.to_latex(index=False, escape=False))
print("Table 3 — Pairwise Statistics:")
print(table3.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 21: Figure 2 — Performance bar chart with error bars (600 DPI)
# ═══════════════════════════════════════════════════════════
summary = pd.read_csv(ARTIFACT_ROOT / "metrics" / "summary_metrics.csv")
MODEL_LABELS = {"medsiglip": "MedSigLIP", "retfound": "RETFound", "efficientnet_b0": "EfficientNet-B0"}
COLORS = {"Dev": "#4472C4", "External": "#ED7D31"}

plot_rows = []
for model_key in ["medsiglip", "retfound", "efficientnet_b0"]:
    b = summary[(summary["task"] == "binary") & (summary["model"] == model_key)].iloc[0]
    m = summary[(summary["task"] == "multiclass") & (summary["model"] == model_key)].iloc[0]
    plot_rows.extend([
        {"Model": MODEL_LABELS[model_key], "Dataset": "Dev", "Metric": "AUC",
         "Value": b["dev_post_auc_mean"], "SD": b["dev_post_auc_sd"]},
        {"Model": MODEL_LABELS[model_key], "Dataset": "External", "Metric": "AUC",
         "Value": b["ext_post_auc_mean"], "SD": b["ext_post_auc_sd"]},
        {"Model": MODEL_LABELS[model_key], "Dataset": "Dev", "Metric": "Macro-F1",
         "Value": m["dev_post_macro_f1_mean"], "SD": m["dev_post_macro_f1_sd"]},
        {"Model": MODEL_LABELS[model_key], "Dataset": "External", "Metric": "Macro-F1",
         "Value": m["ext_post_macro_f1_mean"], "SD": m["ext_post_macro_f1_sd"]},
    ])
plot_df = pd.DataFrame(plot_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric in zip(axes, ["AUC", "Macro-F1"]):
    sub = plot_df[plot_df["Metric"] == metric]
    models = sub["Model"].unique()
    x = np.arange(len(models))
    width = 0.35
    for i, ds in enumerate(["Dev", "External"]):
        ds_sub = sub[sub["Dataset"] == ds].set_index("Model").loc[models]
        bars = ax.bar(x + i * width, ds_sub["Value"].values, width,
                      yerr=ds_sub["SD"].values, capsize=4,
                      color=COLORS[ds], label=ds, edgecolor="white", linewidth=0.5)
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(models, rotation=0)
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.set_ylim(0.0, 1.05)
    ax.legend(loc="lower right")
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Diabetic Retinopathy Benchmark Performance (Dev vs External)", fontsize=14, fontweight="bold")
fig.tight_layout()
for ext in ["png", "pdf", "tiff"]:
    fig.savefig(ARTIFACT_ROOT / "figures" / f"figure2_performance.{ext}", dpi=600)
plt.show()
print("Figure 2 saved (PNG/PDF/TIFF, 600 DPI).")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 22: Figure 3 — Reliability diagrams pre/post TS (600 DPI)
# ═══════════════════════════════════════════════════════════
MODEL_LABELS = {"medsiglip": "MedSigLIP", "retfound": "RETFound", "efficientnet_b0": "EfficientNet-B0"}
MODEL_COLORS = {"medsiglip": "#4472C4", "retfound": "#70AD47", "efficientnet_b0": "#ED7D31"}

def reliability_curve_points(y_true, probs, n_bins=10):
    conf = probs.max(axis=1); pred = probs.argmax(axis=1)
    corr = (pred == y_true).astype(np.float32)
    bins = np.linspace(0, 1, n_bins + 1)
    xs, ys, counts = [], [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.sum() == 0: continue
        xs.append(conf[mask].mean()); ys.append(corr[mask].mean()); counts.append(mask.sum())
    return np.array(xs), np.array(ys), np.array(counts)

binary_stats = {m: np.load(ARTIFACT_ROOT / "metrics" / f"stats_input_binary_{m}.npz")
                for m in ["medsiglip", "retfound", "efficientnet_b0"]}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
panels = [
    ("dev", "pre", axes[0, 0], "Dev — Pre-TS"),
    ("dev", "post", axes[0, 1], "Dev — Post-TS"),
    ("ext", "pre", axes[1, 0], "External — Pre-TS"),
    ("ext", "post", axes[1, 1], "External — Post-TS"),
]
for ds, phase, ax, title in panels:
    for mk in ["medsiglip", "retfound", "efficientnet_b0"]:
        arr = binary_stats[mk]
        x, yhat, _ = reliability_curve_points(arr[f"{ds}_targets"], arr[f"{ds}_probs_{phase}"], n_bins=cfg.ece_bins)
        ax.plot(x, yhat, marker="o", markersize=5, label=MODEL_LABELS[mk], color=MODEL_COLORS[mk], linewidth=2)
    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Perfect calibration")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Predicted confidence")
    ax.set_ylabel("Observed accuracy")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(loc="lower right")
    ax.set_aspect("equal")

fig.suptitle("Reliability Diagrams (Binary Task)", fontsize=14, fontweight="bold")
fig.tight_layout()
for ext in ["png", "pdf", "tiff"]:
    fig.savefig(ARTIFACT_ROOT / "figures" / f"figure3_reliability.{ext}", dpi=600)
plt.show()
print("Figure 3 saved (PNG/PDF/TIFF, 600 DPI).")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 23: Supplementary tables + figures
# ═══════════════════════════════════════════════════════════
supp = ARTIFACT_ROOT / "supplementary"

# ── Table S1: Per-grade breakdown ──
pd.read_csv(ARTIFACT_ROOT / "metrics" / "per_grade_metrics.csv").to_csv(supp / "table_s1_per_grade.csv", index=False)

# ── Table S2: Ablations (placeholder) ──
if cfg.run_ablations:
    print("[TODO] Implement ablation runs (MLP vs linear, class weights)")
else:
    pd.DataFrame([{"status": "skipped", "note": "Set cfg.run_ablations=True"}]).to_csv(supp / "table_s2_ablations.csv", index=False)

# ── Table S3: Seed variance ──
sf = pd.read_csv(ARTIFACT_ROOT / "metrics" / "seed_fold_metrics.csv")
sf.groupby(["task", "model", "seed"]).mean(numeric_only=True).reset_index().to_csv(supp / "table_s3_seed_variance.csv", index=False)

# ── Table S4: External ensemble ──
s4_rows = []
for task_name in ["binary", "multiclass"]:
    for model_key in ["medsiglip", "retfound", "efficientnet_b0"]:
        arr = np.load(ARTIFACT_ROOT / "metrics" / f"stats_input_{task_name}_{model_key}.npz")
        y = arr["ext_targets"]
        pre = compute_metrics(task_name, y, arr["ext_probs_pre"], ece_bins=cfg.ece_bins)
        post = compute_metrics(task_name, y, arr["ext_probs_post"], ece_bins=cfg.ece_bins)
        row = {"task": task_name, "model": model_key}
        row.update({f"pre_{k}": v for k, v in pre.items()})
        row.update({f"post_{k}": v for k, v in post.items()})
        s4_rows.append(row)
pd.DataFrame(s4_rows).to_csv(supp / "table_s4_external_ensemble.csv", index=False)

# ── Table S5: Model profile ──
profile_rows = []
for model_key in ["medsiglip", "retfound", "efficientnet_b0"]:
    bundle = LOADER_REGISTRY[model_key]()
    model = bundle["model"]
    n_params = sum(p.numel() for p in model.parameters())
    bs = 16
    size = MODEL_REGISTRY[model_key]["input_size"]
    if model_key == "medsiglip":
        imgs = [Image.new("RGB", (size, size), (128, 128, 128)) for _ in range(bs)]
        inputs = bundle["processor"](images=imgs, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        torch.cuda.synchronize(); t0 = time.time()
        with torch.inference_mode(), torch.cuda.amp.autocast():
            _ = model.get_image_features(**inputs) if hasattr(model, "get_image_features") else model(**inputs)
        torch.cuda.synchronize(); dt = time.time() - t0
    else:
        x = torch.randn(bs, 3, size, size).to(DEVICE)
        torch.cuda.synchronize(); t0 = time.time()
        with torch.inference_mode(), torch.cuda.amp.autocast():
            _ = model(x)
        torch.cuda.synchronize(); dt = time.time() - t0
    profile_rows.append({"model": model_key, "params": n_params, "batch": bs, "sec": dt, "img_per_sec": bs/max(dt, 1e-9)})
    del bundle; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(profile_rows).to_csv(supp / "table_s5_model_profile.csv", index=False)

# ── Figure S1: Training dynamics ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for task_name in ["binary", "multiclass"]:
    for mk in ["medsiglip", "retfound", "efficientnet_b0"]:
        hist_dir = ARTIFACT_ROOT / "checkpoints" / task_name / mk / "fold_0" / f"seed_{cfg.seeds[0]}"
        hist_files = sorted(hist_dir.glob("*_history.json")) if hist_dir.exists() else []
        if not hist_files: continue
        h = pd.read_json(hist_files[0])
        axes[0].plot(h["epoch"], h["train_loss"], label=f"{task_name}-{mk}")
        axes[1].plot(h["epoch"], h["val_macro_f1"], label=f"{task_name}-{mk}")
axes[0].set_title("Train Loss"); axes[1].set_title("Val Macro-F1")
for ax in axes: ax.set_xlabel("Epoch"); ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(supp / "figure_s1_training_dynamics.png", dpi=600); plt.close(fig)

# ── Figure S2: Confusion matrices ──
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for j, mk in enumerate(["medsiglip", "retfound", "efficientnet_b0"]):
    arr = np.load(ARTIFACT_ROOT / "metrics" / f"stats_input_multiclass_{mk}.npz")
    for i, (sn, y, pr) in enumerate([("Dev", arr["dev_targets"], arr["dev_probs_post"]),
                                      ("Ext", arr["ext_targets"], arr["ext_probs_post"])]):
        cm = confusion_matrix(y, pr.argmax(1), labels=[0,1,2,3,4])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[i, j])
        axes[i, j].set_title(f"{sn} — {MODEL_LABELS.get(mk, mk)}")
        axes[i, j].set_xlabel("Pred"); axes[i, j].set_ylabel("True")
fig.tight_layout()
fig.savefig(supp / "figure_s2_confusion_matrices.png", dpi=600); plt.close(fig)

# ── Figure S3: Error overlap ──
ys = np.load(ARTIFACT_ROOT / "metrics" / "stats_input_binary_medsiglip.npz")["ext_targets"]
correct = {}
for mk in ["medsiglip", "retfound", "efficientnet_b0"]:
    arr = np.load(ARTIFACT_ROOT / "metrics" / f"stats_input_binary_{mk}.npz")
    pred = (arr["ext_probs_post"][:, 1] >= 0.5).astype(int)
    correct[mk] = (pred == ys).astype(int)
patterns = pd.Series([f"M{correct['medsiglip'][i]}_R{correct['retfound'][i]}_E{correct['efficientnet_b0'][i]}"
                       for i in range(len(ys))]).value_counts().reset_index()
patterns.columns = ["pattern", "count"]
patterns.to_csv(supp / "table_s_error_overlap.csv", index=False)
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=patterns, x="pattern", y="count", ax=ax, palette="Set2")
ax.set_title("Error Overlap Patterns (Binary, External)")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(supp / "figure_s3_error_overlap.png", dpi=600); plt.close(fig)

# ── Figure S4: Grade 1 confidence ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, mk in zip(axes, ["medsiglip", "retfound", "efficientnet_b0"]):
    arr = np.load(ARTIFACT_ROOT / "metrics" / f"stats_input_multiclass_{mk}.npz")
    y = arr["ext_targets"]; probs = arr["ext_probs_post"]
    mask = (y == 1); conf = probs[mask].max(axis=1) if mask.sum() > 0 else np.array([0.0])
    sns.histplot(conf, bins=20, ax=ax); ax.set_title(f"{MODEL_LABELS.get(mk, mk)} (Grade 1)"); ax.set_xlabel("Confidence")
fig.tight_layout()
fig.savefig(supp / "figure_s4_grade1_confidence.png", dpi=600); plt.close(fig)

print(f"Supplementary artifacts saved to: {supp}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 24: ZIP export (all artifacts in one bundle)
# ═══════════════════════════════════════════════════════════
bundle_path = ARTIFACT_ROOT / "export" / "DR_benchmark_submission_bundle.zip"

with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in ["figures", "tables", "supplementary", "metrics", "splits",
                    "checkpoints", "predictions", "metadata", "embeddings"]:
        base = ARTIFACT_ROOT / folder
        if not base.exists(): continue
        for fp in base.rglob("*"):
            if fp.is_file():
                zf.write(fp, arcname=str(fp.relative_to(ARTIFACT_ROOT)))

# Validate
required = ["figures", "tables", "metrics"]
with zipfile.ZipFile(bundle_path, "r") as zf:
    names = zf.namelist()
missing = [d for d in required if not any(n.startswith(d + "/") for n in names)]
if missing:
    raise AssertionError(f"ZIP missing: {missing}")

size_mb = bundle_path.stat().st_size / (1024 * 1024)
print(f"Bundle: {bundle_path}")
print(f"Size  : {size_mb:.1f} MB")
print(f"Files : {len(names)}")
print("ZIP validation passed.")

# ── Copy to Drive for persistence ──
import shutil
drive_export = DRIVE_ROOT / "export"
drive_export.mkdir(parents=True, exist_ok=True)
shutil.copy2(bundle_path, drive_export / bundle_path.name)
print(f"Bundle copied to Drive: {drive_export / bundle_path.name}")